# 01 · Data overview

First look at the UCI Heart Disease (Cleveland) dataset: **shape, data types, missing values, and duplicates**.

No cleaning happens here. This notebook only records what the raw data looks like so the cleaning steps in Week 2 are based on evidence.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make the project's src/ package importable from the notebooks/ folder
sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_raw, add_target, RAW_PATH

pd.set_option("display.max_columns", None)
RAW_PATH.name

'processed.cleveland.data'

## 1. Load the data

The raw file has no header row, and missing values are written as `?`. `load_raw()` adds the column names and converts `?` to `NaN`.

In [2]:
df = load_raw()
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 2. Shape

In [3]:
print(f"Rows: {df.shape[0]}  Columns: {df.shape[1]}")

Rows: 303  Columns: 14


## 3. Data types

Everything is read as a number, because pandas can't tell that columns like `cp` or `thal` are category codes. The data dictionary records which ones are categorical.

In [4]:
df.dtypes

age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
num           int64
dtype: object

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


In [6]:
# Number of distinct values per column: low counts suggest a categorical code
df.nunique().sort_values()

sex           2
fbs           2
exang         2
restecg       3
slope         3
thal          3
cp            4
ca            4
num           5
oldpeak      40
age          41
trestbps     50
thalach      91
chol        152
dtype: int64

## 4. Missing values

In [7]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct})[missing > 0]

,missing,pct
ca,4,1.32
thal,2,0.66


In [8]:
# Which rows have a missing value?
df[df.isna().any(axis=1)]

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
87,53.0,0.0,3.0,128.0,216.0,0.0,2.0,115.0,0.0,0.0,1.0,0.0,NaN,0
166,52.0,1.0,3.0,138.0,223.0,0.0,0.0,169.0,0.0,0.0,1.0,NaN,3.0,0
192,43.0,1.0,4.0,132.0,247.0,1.0,2.0,143.0,1.0,0.1,2.0,NaN,7.0,1
266,52.0,1.0,4.0,128.0,204.0,1.0,0.0,156.0,1.0,1.0,2.0,0.0,NaN,2
287,58.0,1.0,2.0,125.0,220.0,0.0,0.0,144.0,0.0,0.4,2.0,NaN,7.0,0
302,38.0,1.0,3.0,138.0,175.0,0.0,0.0,173.0,0.0,0.0,1.0,NaN,3.0,0


## 5. Duplicates

In [9]:
n_dupes = df.duplicated().sum()
print(f"Fully duplicated rows: {n_dupes}")

Fully duplicated rows: 0


## 6. Target variable

`num` ranges from 0 (no disease) to 4. Following standard practice, the binary `target` is 1 whenever `num` > 0.

In [10]:
df["num"].value_counts().sort_index()

num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64

In [11]:
df = add_target(df)
counts = df["target"].value_counts().sort_index()
pd.DataFrame({"count": counts, "pct": (counts / len(df) * 100).round(1)}).rename(index={0: "no disease", 1: "disease"})

,count,pct
target,,
no disease,164,54.1
disease,139,45.9


## 7. Findings

*(Numbers below come from the outputs above.)*

- **Shape:** 303 rows × 14 columns (13 features + `num`).
- **Types:** every column loads as `float64` except `num` (`int64`). Category codes (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `thal`) need to be cast to integer or category types during cleaning.
- **Missing values:** only `ca` (4 rows) and `thal` (2 rows), 6 rows in total (≈2%). Options for Week 2: drop those 6 rows, or fill in the most common value.
- **Duplicates:** none.
- **Target:** fairly balanced (about 54% no disease vs. 46% disease), so accuracy is a reasonable starting metric. Recall still matters most because missing a sick patient is the costlier error.

**Next (Week 2):** handle missing values, fix data types, and start the EDA.